<a href="https://colab.research.google.com/github/venkatshardul/MOVIE-RECOMENDATAION-WITH-ARTIFICIALL-NEURAL-NETWORK/blob/main/Movie_recomendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System

This notebook demonstrates the development of a basic movie recommendation system using PyTorch. The system utilizes an Artificial Neural Network (ANN) with embedding layers to learn representations of users and movies, predicting ratings to suggest new movies to users.

## Table of Contents
1. [Project Objective](#project-objective)
2. [Dataset](#dataset)
3. [Data Loading and Preprocessing](#data-loading-and-preprocessing)
4. [Model Architecture](#model-architecture)
5. [Training and Evaluation](#training-and-evaluation)
6. [Making Recommendations](#making-recommendations)

## Project Objective
To build a movie recommendation system that can predict user ratings for unseen movies and provide personalized movie suggestions based on their historical preferences.

## Dataset
The project uses a small version of the MovieLens dataset (`ml-latest-small`), which contains:
- 100,000 ratings
- 3,600 tag applications
- 9,000 movies
- 600 users

## Data Loading and Preprocessing
1.  **Download and Extraction**: The `ml-latest-small.zip` file is downloaded from GroupLens and extracted.
2.  **DataFrame Loading**: `movies.csv` and `ratings.csv` are loaded into pandas DataFrames.
3.  **Feature Preparation**: The `timestamp` column is dropped from the `ratings_df`.
4.  **Train-Test Split**: The `ratings_df` is split into training and testing sets (80/20 ratio).
5.  **Tensor Conversion**: `userId` and `movieId` are converted to 0-indexed PyTorch tensors, and ratings are converted to float32 tensors.
6.  **DataLoader Creation**: `TensorDataset` and `DataLoader` are used to prepare the data for batch processing during training.

## Model Architecture
The recommendation system uses an Artificial Neural Network (`MovieRecommender`) with the following structure:
-   **Embedding Layers**: Separate embedding layers for `userId` and `movieId`, each mapping to a 50-dimensional vector.
-   **Concatenation**: The user and movie embeddings are concatenated.
-   **Dense Layers**: A sequential model with three linear layers and ReLU activation functions:
    -   Input layer: `embedding_dim * 2` (100) neurons
    -   Hidden layer 1: 128 neurons, ReLU activation
    -   Hidden layer 2: 64 neurons, ReLU activation
    -   Output layer: 1 neuron (for predicted rating)

## Training and Evaluation
-   **Optimizer**: Adam optimizer with a learning rate of 0.001.
-   **Loss Function**: Mean Squared Error (MSELoss).
-   **Epochs**: The model is trained for 10 epochs.
-   **Monitoring**: Training and validation (test) loss are tracked per epoch.
-   **Evaluation Metric**: Root Mean Squared Error (RMSE) is used to evaluate the model's performance on the test set.

## Making Recommendations
A `recommend_movies` function is provided to generate personalized movie recommendations for any given `userId`:
1.  **Identify Unrated Movies**: It first determines which movies a user has not yet rated.
2.  **Predict Ratings**: For these unrated movies, it uses the trained model to predict potential ratings.
3.  **Merge Information**: Predicted ratings are merged with movie details (title, genres).
4.  **Sort and Recommend**: The movies are then sorted by their predicted ratings in descending order, and the top N recommendations are returned.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


### Loading MovieLens Dataset

I will download and load a small version of the MovieLens dataset (ml-latest-small) from the GroupLens website. This dataset contains 100,000 ratings and 3,600 tag applications applied to 9,000 movies by 600 users.

First, I will download the dataset as a zip file, extract its contents, and then load the `movies.csv` and `ratings.csv` files into pandas DataFrames.

In [ ]:
import requests
import zipfile
import io
import os

# Download the MovieLens dataset
url = "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
response = requests.get(url)

# Extract the zip file contents
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("movielens_data")

# Load movies.csv and ratings.csv into pandas DataFrames
movies_df = pd.read_csv("movielens_data/ml-latest-small/movies.csv")
ratings_df = pd.read_csv("movielens_data/ml-latest-small/ratings.csv")

print("Movies DataFrame head:")
display(movies_df.head())

print("\nRatings DataFrame head:")
display(ratings_df.head())

Movies DataFrame head:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Ratings DataFrame head:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
movies_df.head

<bound method NDFrame.head of       movieId                                      title  \
0           1                           Toy Story (1995)   
1           2                             Jumanji (1995)   
2           3                    Grumpier Old Men (1995)   
3           4                   Waiting to Exhale (1995)   
4           5         Father of the Bride Part II (1995)   
...       ...                                        ...   
9737   193581  Black Butler: Book of the Atlantic (2017)   
9738   193583               No Game No Life: Zero (2017)   
9739   193585                               Flint (2017)   
9740   193587        Bungo Stray Dogs: Dead Apple (2018)   
9741   193609        Andrew Dice Clay: Dice Rules (1991)   

                                           genres  
0     Adventure|Animation|Children|Comedy|Fantasy  
1                      Adventure|Children|Fantasy  
2                                  Comedy|Romance  
3                            Comedy|Drama|Romance  
4                                          Comedy  
...                                           ...  
9737              Action|Animation|Comedy|Fantasy  
9738                     Animation|Comedy|Fantasy  
9739                                        Drama  
9740                             Action|Animation  
9741                                       Comedy  

[9742 rows x 3 columns]>

In [ ]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
ratings_df.drop("timestamp",axis=1,inplace=True)

In [ ]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [ ]:
x=ratings_df.drop("rating",axis=1)
y=ratings_df["rating"]

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
x_train.isnull().sum()

,0
userId,0
movieId,0


In [ ]:
y_train.isnull().sum()

np.int64(0)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
max_user_id = ratings_df['userId'].max()
max_movie_id = ratings_df['movieId'].max()

user_train_tensor = torch.tensor(x_train['userId'].values - 1, dtype=torch.long)
movie_train_tensor = torch.tensor(x_train['movieId'].values - 1, dtype=torch.long)
user_test_tensor = torch.tensor(x_test['userId'].values - 1, dtype=torch.long)
movie_test_tensor = torch.tensor(x_test['movieId'].values - 1, dtype=torch.long)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

In [ ]:
movie_train_tensor

tensor([ 7346, 71461,  2114,  ...,  6866,   980,  6710])

In [ ]:
train_dataset = TensorDataset(user_train_tensor, movie_train_tensor, y_train_tensor)
test_dataset = TensorDataset(user_test_tensor, movie_test_tensor, y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

ANN

In [ ]:
class MovieRecommender(nn.Module):
  def __init__(self, num_users, num_movies, embedding_dim=50):
    super(MovieRecommender,self).__init__()
    self.user_embedding = nn.Embedding(num_users, embedding_dim)
    self.movie_embedding = nn.Embedding(num_movies, embedding_dim)
    self.model=nn.Sequential(
        nn.Linear(embedding_dim * 2, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    )

  def forward(self, user_input, movie_input):
    user_embedded = self.user_embedding(user_input)
    movie_embedded = self.movie_embedding(movie_input)
    x=torch.cat([user_embedded,movie_embedded],dim=1)
    return self.model(x).squeeze()

In [ ]:
num_users = int(max_user_id + 1)
num_movies = int(max_movie_id + 1)

model=MovieRecommender(num_users=num_users, num_movies=num_movies)
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs=10
train_loss=[]
val_loss=[]

for epoch in range(epochs):
  model.train()
  train_loss_epoch=[]
  for users,movies,ratings in train_loader:
    optimizer.zero_grad()
    predictions=model(users,movies)
    loss=criterion(predictions,ratings)
    loss.backward()
    optimizer.step()
    avg_train_loss=loss.item()
    train_loss_epoch.append(avg_train_loss)
  train_loss.append(np.mean(train_loss_epoch))

  print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss[-1]:.4f}")

  model.eval()
  test_loss=0.0
  with torch.no_grad():
    for users,movies,ratings in test_loader:
      predictions=model(users,movies)
      loss=criterion(predictions,ratings)
      test_loss+=loss.item()

    avg_test_loss=test_loss/len(test_loader)
    val_loss.append(avg_test_loss)
    print(f"Epoch {epoch+1}/{epochs}, Test Loss: {avg_test_loss:.4f}")

Epoch 1/10, Train Loss: 1.1262
Epoch 1/10, Test Loss: 0.9213
Epoch 2/10, Train Loss: 0.8228
Epoch 2/10, Test Loss: 0.8595
Epoch 3/10, Train Loss: 0.7580
Epoch 3/10, Test Loss: 0.8330
Epoch 4/10, Train Loss: 0.7111
Epoch 4/10, Test Loss: 0.8261
Epoch 5/10, Train Loss: 0.6753
Epoch 5/10, Test Loss: 0.8221
Epoch 6/10, Train Loss: 0.6445
Epoch 6/10, Test Loss: 0.8183
Epoch 7/10, Train Loss: 0.6130
Epoch 7/10, Test Loss: 0.8249
Epoch 8/10, Train Loss: 0.5851
Epoch 8/10, Test Loss: 0.8368
Epoch 9/10, Train Loss: 0.5558
Epoch 9/10, Test Loss: 0.8699
Epoch 10/10, Train Loss: 0.5244
Epoch 10/10, Test Loss: 0.8664


In [ ]:
from sklearn.metrics import mean_squared_error

model.eval()
all_predictions = []
all_actuals = []

with torch.no_grad():
    for users, movies, ratings in test_loader:
        predictions = model(users, movies)
        all_predictions.extend(predictions.cpu().numpy())
        all_actuals.extend(ratings.cpu().numpy())

rmse = np.sqrt(mean_squared_error(all_actuals, all_predictions))
print(f"Test RMSE: {rmse:.4f}")

Test RMSE: 0.9318


In [ ]:
# Merge movie information with ratings data
merged_df = pd.merge(ratings_df, movies_df, on='movieId')
display(merged_df.head())

,userId,movieId,rating,title,genres
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [ ]:
def recommend_movies(user_id, model, movies_df, ratings_df, num_recommendations=10):

    user_rated_movies = ratings_df[ratings_df['userId'] == user_id]['movieId'].unique()


    all_movie_ids = movies_df['movieId'].unique()


    unrated_movie_ids = np.setdiff1d(all_movie_ids, user_rated_movies)


    user_tensor = torch.tensor([user_id - 1] * len(unrated_movie_ids), dtype=torch.long)
    movie_tensor = torch.tensor(unrated_movie_ids - 1, dtype=torch.long)


    model.eval()
    with torch.no_grad():
        predicted_ratings = model(user_tensor, movie_tensor)


    predictions_df = pd.DataFrame({
        'movieId': unrated_movie_ids,
        'predicted_rating': predicted_ratings.cpu().numpy()
    })


    predictions_df = pd.merge(predictions_df, movies_df, on='movieId')

    top_recommendations = predictions_df.sort_values(by='predicted_rating', ascending=False)

    return top_recommendations.head(num_recommendations)

In [ ]:
# Example usage: Get recommendations for user 1
user_id_to_recommend = 6
recommendations = recommend_movies(user_id_to_recommend, model, movies_df, ratings_df, num_recommendations=10)

print(f"Top 10 movie recommendations for User {user_id_to_recommend}:")
display(recommendations)

Top 10 movie recommendations for User 6:


,movieId,predicted_rating,title,genres
8644,136834,5.607844,The Eye: Infinity (2005),Horror
8511,131610,5.603857,Willy/Milly (1986),Comedy|Fantasy
8936,155743,5.501739,My Big Fat Greek Wedding 2 (2016),Comedy
365,858,5.434476,"Godfather, The (1972)",Crime|Drama
8352,121231,5.376823,It Follows (2014),Horror
3795,5889,5.352980,"Cruel Romance, A (Zhestokij Romans) (1984)",Drama|Romance
8525,132333,5.318511,Seve (2014),Documentary|Drama
7,28,5.265608,Persuasion (1995),Drama|Romance
7307,87234,5.264329,Submarine (2010),Comedy|Drama|Romance
8317,118930,5.263859,Bill Burr: I'm Sorry You Feel That Way (2014),Comedy
